#### Step 1 — Locate the Meadowlark SDK and inspect the `slmsuite` backend

This is a **read-only diagnostic test**. It does not initialize the SLM or send any pattern to it.

The cell checks:

* which Python interpreter and `slmsuite` installation are being used,
* whether `slmsuite` contains a Meadowlark/Blink hardware backend,
* where the Meadowlark backend source file is located,
* what SDK/DLL files that backend expects,
* where Meadowlark/Blink is installed on Windows,
* which relevant `.dll`, `.h`, `.lib`, and Python files are present.

This establishes whether the required **Blink/Meadowlark SDK components are installed and visible to `slmsuite`** before attempting any hardware communication.


In [1]:
# STEP 1 — Locate Blink/Meadowlark installation and inspect slmsuite's Meadowlark backend
# Read-only: this does NOT initialize or write anything to the SLM.

from pathlib import Path
import os
import sys
import pkgutil
import inspect

print("=== Python ===")
print("Executable:", sys.executable)
print("Version:", sys.version)
print()

# ------------------------------------------------------------
# 1. Locate slmsuite and search for Meadowlark-related modules
# ------------------------------------------------------------

import slmsuite

print("=== slmsuite ===")
print("slmsuite package:", Path(slmsuite.__file__).resolve())
print()

print("=== Searching slmsuite modules for Meadowlark/Blink ===")

matching_modules = []

for module_info in pkgutil.walk_packages(
    slmsuite.__path__,
    prefix="slmsuite."
):
    name = module_info.name.lower()

    if "meadow" in name or "blink" in name:
        matching_modules.append(module_info.name)

if matching_modules:
    for name in matching_modules:
        print("FOUND:", name)
else:
    print("No module name containing 'meadow' or 'blink' was found.")

print()

# ------------------------------------------------------------
# 2. Import matching modules and show their source files
# ------------------------------------------------------------

print("=== Meadowlark/Blink module source locations ===")

loaded_modules = []

for module_name in matching_modules:
    try:
        module = __import__(module_name, fromlist=["*"])
        loaded_modules.append(module)

        print("\nMODULE:", module_name)

        try:
            print("FILE:", Path(inspect.getfile(module)).resolve())
        except Exception as e:
            print("Could not determine source file:", repr(e))

    except Exception as e:
        print("\nCould not import:", module_name)
        print("ERROR:", repr(e))

print()

# ------------------------------------------------------------
# 3. Print source lines mentioning SDK/DLL/path information
# ------------------------------------------------------------

print("=== Relevant slmsuite source lines ===")

keywords = (
    "dll",
    "sdk",
    "blink",
    "meadowlark",
    "path",
    "loadlibrary",
    "ctypes",
)

for module in loaded_modules:
    try:
        source = inspect.getsource(module)
    except Exception as e:
        print(f"\nCould not inspect {module.__name__}: {e}")
        continue

    print(f"\n--- {module.__name__} ---")

    found_any = False

    for lineno, line in enumerate(source.splitlines(), start=1):
        if any(k in line.lower() for k in keywords):
            print(f"{lineno:4d}: {line}")
            found_any = True

    if not found_any:
        print("No relevant SDK/DLL/path lines found.")

print()

# ------------------------------------------------------------
# 4. Look for likely installed Meadowlark/Blink directories
#    without assuming an SDK filename
# ------------------------------------------------------------

print("=== Windows installation candidates ===")

roots = []

for env_name in (
    "ProgramFiles",
    "ProgramFiles(x86)",
    "ProgramData",
    "LOCALAPPDATA",
):
    value = os.environ.get(env_name)
    if value:
        roots.append(Path(value))

candidates = []

for root in roots:
    if not root.exists():
        continue

    # Only inspect the first directory level so this remains fast/read-only.
    try:
        for item in root.iterdir():
            name = item.name.lower()

            if (
                "meadow" in name
                or "blink" in name
            ):
                candidates.append(item.resolve())
    except PermissionError:
        pass

if candidates:
    for path in candidates:
        print(path)
else:
    print("No obvious top-level Meadowlark/Blink directory found.")

print()

# ------------------------------------------------------------
# 5. If candidate directories exist, list SDK-like files
#    without assuming exact filenames
# ------------------------------------------------------------

print("=== SDK-like files inside candidates ===")

extensions = {
    ".dll",
    ".lib",
    ".h",
    ".hpp",
    ".py",
    ".pyd",
}

for base in candidates:
    print(f"\n--- {base} ---")

    try:
        files = [
            p for p in base.rglob("*")
            if p.is_file() and p.suffix.lower() in extensions
        ]

        if not files:
            print("No DLL/LIB/header/Python extension files found.")
        else:
            for p in files[:200]:
                print(p)

            if len(files) > 200:
                print(f"... {len(files) - 200} additional files omitted")

    except PermissionError:
        print("Permission denied while scanning this directory.")

=== Python ===
Executable: c:\Users\admin\AppData\Local\Python\pythoncore-3.12-64\python.exe
Version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

=== slmsuite ===
slmsuite package: C:\Users\admin\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\slmsuite\__init__.py

=== Searching slmsuite modules for Meadowlark/Blink ===
FOUND: slmsuite.hardware.slms.meadowlark

=== Meadowlark/Blink module source locations ===

MODULE: slmsuite.hardware.slms.meadowlark
FILE: C:\Users\admin\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\slmsuite\hardware\slms\meadowlark.py

=== Relevant slmsuite source lines ===

--- slmsuite.hardware.slms.meadowlark ---
   2: Hardware control for Meadowlark SLMs.
   4: Meadowlark distributes several different interfaces for their products.
   5: The following versions are supported in slmsuite at Meadowlark's suggestion
   6: (contact Meadowlark to upgrade to one of these versions):
   8: .. csv-table:: Meadow

c:\Users\admin\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\admin\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\slmsuite\holography\algorithms\_header.py:30: UserWarning: cupy is not installed; using numpy. Install cupy for faster GPU-based holography.
  warnings.warn(


#### Step 2 — Verify and classify the installed Meadowlark SDK

This is still a **read-only test**. It does not initialize the SLM or send anything to the hardware.

The cell checks the two Meadowlark SDK folders found in Step 1 and verifies that each contains:

* `Blink_C_wrapper.dll`
* `Blink_C_wrapper.h`

It then uses `slmsuite`'s own `_parse_header()` function to determine:

* which Meadowlark SDK mode is installed,
* whether it is recognized as an **HDMI SDK**,
* the exact DLL path that would be used,
* the detected SDK API signature (`SDK trace`).

This confirms that the installed Blink SDK is compatible with the Meadowlark HDMI backend in the current `slmsuite` version.


In [2]:
from pathlib import Path
from slmsuite.hardware.slms.meadowlark import Meadowlark, _SDK_MODE_NAMES

folders = [
    Path(r"C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK"),
    Path(r"C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\Cal Kit"),
]

for folder in folders:
    print("\n" + "=" * 80)
    print("Testing:", folder)

    dll = folder / "Blink_C_wrapper.dll"
    header = folder / "Blink_C_wrapper.h"

    print("DLL exists:   ", dll.exists())
    print("Header exists:", header.exists())

    try:
        mode, dll_path, trace = Meadowlark._parse_header(str(folder), warn=True)

        print("Detected mode :", mode)
        print("Mode name     :", _SDK_MODE_NAMES.get(mode, "UNKNOWN"))
        print("DLL path      :", dll_path)
        print("SDK trace     :", trace)

    except Exception as e:
        print("ERROR:", type(e).__name__, str(e))


Testing: C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK
DLL exists:    True
Header exists: True
Detected mode : 1
Mode name     : HDMI
DLL path      : C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK\Blink_C_wrapper.dll
SDK trace     : (0, 2)

Testing: C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\Cal Kit
DLL exists:    True
Header exists: True
Detected mode : 1
Mode name     : HDMI
DLL path      : C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\Cal Kit\Blink_C_wrapper.dll
SDK trace     : (0, 2)


#### Step 3 — Detect the physical SLM through the Meadowlark SDK

This is the first test that **loads the Meadowlark SDK and communicates with the connected hardware**, but it does not yet create an SLM control object or write a phase pattern.

The cell uses `Meadowlark.info()` with the explicitly selected SDK path to:

* load the `Blink_C_wrapper.dll`,
* initialize the Meadowlark HDMI SDK,
* detect the connected SLM,
* report its resolution and bit depth,
* confirm that `slmsuite` can see the physical device through the SDK.

A successful result confirms the communication chain:

`Jupyter → slmsuite → Meadowlark SDK → physical HDMI SLM`


In [3]:
from slmsuite.hardware.slms.meadowlark import Meadowlark

sdk_path = r"C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK"

devices = Meadowlark.info(
    verbose=True,
    sdk_path=sdk_path
)

print("\nReturned object:")
print(devices)

Using HDMI SDK at 'C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK\Blink_C_wrapper.dll'
SLM 1: Meadowlark HDMI (1920x1200, 8-bit)

Returned object:
[(1, 'Meadowlark HDMI (1920x1200, 8-bit)')]


#### Step 4 — Verify the selected SDK and 759 nm LUT paths

This is a **read-only configuration check** before creating the SLM control object.

The cell verifies that:

* the chosen Meadowlark SDK folder exists,
* the selected 759 nm LUT file exists,
* the exact LUT path is correct,
* the installed `Meadowlark` class accepts the expected initialization arguments.

Printing the constructor signature confirms that `slmsuite` can be initialized with explicit `sdk_path`, `lut_path`, and wavelength parameters before we connect the SLM object.


In [4]:
from pathlib import Path
import inspect
from slmsuite.hardware.slms.meadowlark import Meadowlark

sdk_path = r"C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK"
lut_path = r"C:\Users\admin\Desktop\Projects\SLM\LUT from Fan\759nm_roomtemp_global\759nm_roomtemp_global\759nm_rommtemp_global.lut"

print("SDK path exists :", Path(sdk_path).exists())
print("LUT path exists :", Path(lut_path).exists())
print("LUT path        :", lut_path)
print()
print("Meadowlark constructor signature:")
print(inspect.signature(Meadowlark))

SDK path exists : True
LUT path exists : True
LUT path        : C:\Users\admin\Desktop\Projects\SLM\LUT from Fan\759nm_roomtemp_global\759nm_roomtemp_global\759nm_rommtemp_global.lut

Meadowlark constructor signature:
(slm_number: int = 1, sdk_path: Optional[str] = None, lut_path: Optional[str] = None, wav_um: float = 1, pitch_um: Optional[Tuple[float, float]] = None, verbose: bool = True, **kwargs)


#### Step 5 — Verify how wavelength and LUT are handled during initialization

This is a **read-only source-code inspection** of the installed `slmsuite` Meadowlark class.

The cell inspects the actual `Meadowlark.__init__()` implementation and prints only the lines related to:

* `wav_um`,
* `lut_path`,
* `load_lut()`,
* `pitch_um`,
* the call to the parent `SLM` class.

The purpose is to confirm that `wav_um` is passed into the base SLM class and that the specified LUT is explicitly loaded during initialization, rather than relying on assumptions about another `slmsuite` version.


In [5]:
import inspect
from slmsuite.hardware.slms.meadowlark import Meadowlark

source = inspect.getsource(Meadowlark.__init__)

for i, line in enumerate(source.splitlines(), start=1):
    if any(key in line.lower() for key in [
        "wav_um",
        "lut_path",
        "super(",
        "load_lut",
        "pitch_um",
    ]):
        print(f"{i:3d}: {line}")

  5:         lut_path: Optional[str] = None,
  6:         wav_um: float = 1,
  7:         pitch_um: Optional[Tuple[float, float]] = None,
 36:         lut_path : str OR None
 37:             Passed to :meth:`load_lut`. Looks for the voltage 'look-up table' data
 42:             See :meth:`load_lut` for how the default
 44:         wav_um : float
 46:         pitch_um : (float, float) OR None
112:         true_lut_path = self.load_lut(lut_path)
113:         if verbose and true_lut_path != lut_path:
114:             print(f"success\n(loaded from '{true_lut_path}')")
121:         super().__init__(
128:             wav_um=wav_um,
129:             pitch_um=(pitch_um if pitch_um else Meadowlark._get_pitch(self.sdk_mode, self.slm_number)),


#### Step 6 — Read the Meadowlark initialization documentation

This is a **read-only documentation check**.

The cell prints the docstring for `Meadowlark.__init__()` from the installed `slmsuite` version. This confirms the intended meaning of the initialization parameters, especially:

* `sdk_path` — path to the Blink SDK installation,
* `lut_path` — LUT file used to run the SLM,
* `wav_um` — operating wavelength in microns,
* `pitch_um` — SLM pixel pitch in microns.

For 759 nm operation, the documentation confirms that the appropriate wavelength argument is `wav_um = 0.759`.


In [6]:
print(inspect.getdoc(Meadowlark.__init__))

Initializes an instance of a Meadowlark SLM.

Arguments
---------
verbose : bool
    Whether to print extra information.
slm_number : int
    The board number of the SLM to connect to,
    in the case of multiple PCIe SLMs. Defaults to 1.
    Ignored for HDMI SLMs when the suggested 1.1.4.120 version of the SDK is
    used. For non-standard SDK version (for example in Blink 1.1.4.124),
    the `slm_number` argument may be used if C header requires.
sdk_path : str
    Path of the Blink SDK installation folder.

    Important
    ~~~~~~~~~
    If the installation is not in the default folder,
    then this path needs to be specified. If there are multiple installations,
    then the most recent installation is chosen. The user must further specify
    the path otherwise. Keep in mind that different versions of the SDK may not
    be compatible with given hardware (HDMI, PCIe, etc.).
    See the compatibility table at
    :module:`slmsuite.hardware.slms.meadowlark` for more information.
l

#### Step 7 — Initialize the Meadowlark SLM with the 759 nm configuration

This is the first step that creates the actual `slm` control object.

The cell initializes the Meadowlark SLM using:

* the explicitly selected Blink SDK folder,
* the selected 759 nm LUT file,
* `wav_um = 0.759`,
* SLM number 1.

During initialization, `slmsuite`:

* validates the Windows display/DPI configuration,
* loads the Meadowlark HDMI SDK,
* loads the specified LUT,
* reads the SLM properties from the hardware,
* creates the Python object used for all later phase writes.

The printed information confirms the detected SLM name, resolution, bit depth, wavelength, pixel pitch, SDK mode, DLL path, and LUT being used.

A successful result means the control chain is fully initialized:

`Jupyter → slmsuite → Meadowlark HDMI SDK → 759 nm LUT → physical SLM`


In [7]:
from slmsuite.hardware.slms.meadowlark import Meadowlark, _SDK_MODE_NAMES

sdk_path = r"C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK"

lut_path = (
    r"C:\Users\admin\Desktop\Projects\SLM\LUT from Fan"
    r"\759nm_roomtemp_global\759nm_roomtemp_global"
    r"\759nm_rommtemp_global.lut"
)

slm = Meadowlark(
    slm_number=1,
    sdk_path=sdk_path,
    lut_path=lut_path,
    wav_um=0.759,
    verbose=True,
)

print("\n=== Initialized SLM ===")
print("Name       :", slm.name)
print("Shape      :", slm.shape)
print("Bit depth  :", slm.bitdepth)
print("Wavelength :", slm.wav_um, "um")
print("Pixel pitch:", slm.pitch_um, "um")
print("SDK mode   :", _SDK_MODE_NAMES[slm.sdk_mode])
print("SDK DLL    :", Meadowlark._sdk_path[slm.sdk_mode])
print("LUT        :", lut_path)

Validating DPI awareness...success
Constructing Blink SDK...success
Loading LUT file...
=== Initialized SLM ===
Name       : Meadowlark HDMI
Shape      : (1200, 1920)
Bit depth  : 8
Wavelength : 0.759 um
Pixel pitch: [8. 8.] um
SDK mode   : HDMI
SDK DLL    : C:\Program Files\Meadowlark Optics\Blink 1920 HDMI\SDK\Blink_C_wrapper.dll
LUT        : C:\Users\admin\Desktop\Projects\SLM\LUT from Fan\759nm_roomtemp_global\759nm_roomtemp_global\759nm_rommtemp_global.lut
